# 🔬 Notebook 11: 模型可解释性 — SHAP & 系数分析

## Learning Objectives / 本节目标
1. 理解 SHAP (SHapley Additive exPlanations) 的核心原理
2. 使用 SHAP TreeExplainer 解释 XGBoost 的单次预测
3. 使用 SHAP LinearExplainer 分析 LEAR Lasso 模型的系数
4. 对比树模型与线性模型对不同特征的依赖差异
5. 掌握跨模型特征重要性排名表的解读方法

## 背景知识

### 为什么需要模型可解释性？

在电力交易中，模型不仅要预测准确，还要解释**为什么**做出某个预测：
- 监管要求：电力市场报价需要有合理解释
- 风控需求：知道哪些因素驱动了极端预测值
- 模型改进：发现特征工程的盲区

### SHAP 核心思想

SHAP 基于博弈论中的 Shapley 值，将预测值分解为每个特征的边际贡献：

```
prediction = base_value + Σ shap_value_i
```

- **base_value**: 模型在所有训练样本上的平均预测值
- **shap_value_i**: 第 i 个特征对这个样本的贡献（可正可负）

### 两种 Explainer

| Explainer | 适用模型 | 原理 |
|-----------|----------|------|
| **TreeExplainer** | XGBoost, RandomForest | 遍历树结构，精确计算 Shapley 值 |
| **LinearExplainer** | Lasso, LinearRegression | 利用系数和特征值直接计算 |

> 💡 **思考**: SHAP 值为什么比简单的特征重要性更有信息量？
> 特征重要性只告诉你特征「重不重要」，
> 而 SHAP 值告诉你「在某个特定样本上，特征推高了还是拉低了预测值」。


In [ ]:
# ── 导入依赖 ──────────────────────────────────
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
from plotly.subplots import make_subplots

pio.renderers.default = 'notebook'

from pipeline.forecaster import XGBoostForecaster
from pipeline.price_forecaster import LEARForecaster
from pipeline.shap_explainer import (
    explain_xgboost_sample,
    explain_lear_sample,
    feature_importance_ranking,
)

print('All imports OK')


---
## Step 1: 加载数据和已训练模型

首先加载 Phase 1 清洗后的负荷数据，然后加载已训练的 XGBoost 和 LEAR 模型。
如果模型文件不存在，会提示先运行对应的训练 notebook。

**模型文件路径约定:**
- XGBoost: `ellectric/models/xgboost_load.joblib`
- LEAR: `ellectric/models/lear_price.joblib`


In [ ]:
import os
import joblib
from pathlib import Path

# ── 路径 ────────────────────────────
MODEL_DIR = Path('..') / 'models'
DATA_DIR = Path('..') / 'data'

# ── 加载特征数据 ────────────────────
data_candidates = [
    DATA_DIR / 'cleaned_load.parquet',
    DATA_DIR / 'engineered_features.parquet',
    DATA_DIR / 'processed' / 'engineered_features.parquet',
]

X = None
for path in data_candidates:
    if path.exists():
        X = pd.read_parquet(path)
        print(f'Loaded features from: {path}  ({len(X)} rows, {len(X.columns)} cols)')
        break

if X is None:
    print(
        '⚠️  特征数据未找到。请先运行 Notebook 03 (Feature Engineering) 生成数据。\n'
        '    未找到路径: '
        + ', '.join(str(p) for p in data_candidates)
    )
else:
    # 只保留数值列，删除索引和日期列
    drop_cols = [c for c in ['timestamp', 'time', 'date', 'ds', 'target'] if c in X.columns]
    if drop_cols:
        print(f'Dropping non-feature columns: {drop_cols}')
        X = X.drop(columns=drop_cols)
    print(f'Feature matrix shape: {X.shape}')


In [ ]:
# ── 加载 XGBoost 模型 ───────────────
xgb = None
xgb_path = MODEL_DIR / 'xgboost_load.joblib'
if xgb_path.exists():
    with open(xgb_path, 'rb') as f:
        xgb = pickle.load(f)
    print(f'XGBoost model loaded: {xgb_path}')
    print(f'  Feature columns: {len(getattr(xgb, "_feature_cols", []))}')
    if getattr(xgb, '_model', None) is not None:
        print(f'  Model trained: YES (type={type(xgb._model).__name__})')
    else:
        print(f'  Model trained: NO')
else:
    print('⚠️  XGBoost 模型未找到。请先运行 Notebook 04 训练模型。')
    print(f'    预期路径: {xgb_path}')


In [ ]:
# ── 加载 LEAR 模型 ────────────────
lear = None
lear_path = MODEL_DIR / 'lear_price.joblib'
if lear_path.exists():
    with open(lear_path, 'rb') as f:
        lear = pickle.load(f)
    print(f'LEAR model loaded: {lear_path}')
    print(f'  Feature columns: {len(getattr(lear, "_feature_cols", []))}')
    if getattr(lear, '_model', None) is not None:
        print(f'  Model trained: YES (type={type(lear._model).__name__})')
    else:
        print(f'  Model trained: NO')
else:
    print('⚠️  LEAR 模型未找到。请先运行 Notebook 06 训练模型。')
    print(f'    预期路径: {lear_path}')


---
## Step 2: XGBoost SHAP Waterfall — 单样本解释

选择一条测试样本（默认第 0 条），用 SHAP TreeExplainer 计算每个特征的贡献，
绘制 waterfall 图：蓝色表示推高预测值（正贡献），红色表示拉低预测值（负贡献）。

### 理解 Waterfall 图

```
base_value (所有样本的平均预测值)
    + feature_A 的贡献 (+Δ)     → 推高预测
    + feature_B 的贡献 (-Δ)     → 拉低预测
    + ...
    = 最终 prediction
```

你可以修改 `sample_idx` 来查看不同样本的解释结果。


In [ ]:
# ── XGBoost SHAP Waterfall ───────────
if xgb is not None and X is not None and hasattr(xgb, '_feature_cols'):
    sample_idx = 0  # 修改此值可查看不同样本
    try:
        fig_xgb = explain_xgboost_sample(
            model=xgb,
            X=X,
            sample_idx=sample_idx,
            max_display=12,
        )
        fig_xgb.show()
    except Exception as e:
        print(f'❌ SHAP 计算失败: {e}')
        print('  可能原因: 特征列不匹配或模型未训练')
else:
    print('⚠️  模型或数据未加载，无法生成 SHAP Waterfall。')
    print('  先确保以上单元格都已成功执行。')


---
## Step 3: XGBoost SHAP Summary Bar — 全局特征重要性

Waterfall 只展示了一个样本，现在我们看全局：
在所有样本上平均的 |SHAP 值| 反映了特征的整体重要性。

**与 XGBoost 内置 feature_importances_ 的区别:**
- 内置重要性: 基于特征被选作分裂点的次数或信息增益
- SHAP 重要性: 基于特征对预测的实际影响（正/负方向都考虑）
- SHAP 更精确，但计算成本更高


In [ ]:
# ── XGBoost SHAP Summary Bar ─────────
if xgb is not None and X is not None and hasattr(xgb, '_feature_cols'):
    import shap as _shap

    feature_cols = xgb._feature_cols
    missing = [c for c in feature_cols if c not in X.columns]
    if missing:
        print(f'⚠️  X 中缺少特征列: {missing[:5]}... (共 {len(missing)} 个)')
    else:
        X_sub = X[feature_cols].copy()

        n_samples = min(1000, len(X_sub))
        X_sample = X_sub.iloc[:n_samples]
        print(f'Computing SHAP values on {n_samples} samples...')

        explainer = _shap.TreeExplainer(xgb._model)
        shap_values = explainer.shap_values(X_sample)

        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        sort_idx = np.argsort(mean_abs_shap)[::-1]

        top_n = min(10, len(feature_cols))
        top_features = [feature_cols[i] for i in sort_idx[:top_n]]
        top_values = [float(mean_abs_shap[i]) for i in sort_idx[:top_n]]

        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=top_values, y=top_features, orientation='h',
            marker_color='#2ca02c',
            text=[f'{v:.4f}' for v in top_values],
            textposition='outside',
        ))
        fig.update_layout(
            title=dict(
                text=f'XGBoost SHAP Summary Bar — Top {top_n} Features'
                     f'<br><sup>Based on {n_samples} samples, mean |SHAP value|</sup>',
                font=dict(size=16),
            ),
            xaxis=dict(title='Mean |SHAP Value|'),
            yaxis=dict(title='Feature', autorange='reversed'),
            height=400,
            margin=dict(l=160, r=40, t=80, b=40),
        )
        fig.show()
else:
    print('⚠️  模型或数据未加载。')


---
## Step 4: LEAR 系数分析 — Lasso 的稀疏解

与 XGBoost 不同，Lasso 线性模型的系数有直观的物理解释：
- 系数的**正负号**表示特征与目标的正/负相关关系
- 系数的**绝对值**表示特征的边际影响大小
- L1 正则化会将不重要特征的系数压缩到**正好为零**

这是 LEAR 的核心优势：自动特征选择 + 可解释性。

**在电价预测中典型的系数模式:**
- `lag24_price` 系数 ≈ +0.5 → 昨天此时电价高，今天也高（持续性）
- `hour_peak` 系数 ≈ +20 → 峰时电价显著高于均值
- `load_lag24` 系数 = 0 → 负荷信息没有额外预测价值（已被价格滞后吸收）


In [ ]:
# ── LEAR 系数分析 ────────────────
if lear is not None and hasattr(lear, '_model') and lear._model is not None:
    coef = lear._model.coef_
    feature_cols = getattr(lear, '_feature_cols', None)
    if feature_cols is None:
        feature_cols = [f'feature_{i}' for i in range(len(coef))]

    zero_count = np.sum(np.abs(coef) < 1e-10)
    total = len(coef)
    print(f'LEAR Lasso 系数分析:')
    print(f'  总特征数: {total}')
    print(f'  零系数: {zero_count} ({zero_count/total*100:.1f}%)')
    print(f'  非零系数: {total - zero_count} ({(total-zero_count)/total*100:.1f}%)')

    abs_coef = np.abs(coef)
    sort_idx = np.argsort(abs_coef)[::-1]
    top_n = min(15, len(coef))

    top_names = [feature_cols[i] for i in sort_idx[:top_n]]
    top_coef = [float(coef[i]) for i in sort_idx[:top_n]]
    top_colors = ['#d62728' if c > 0 else '#1f77b4' for c in top_coef]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=top_coef, y=top_names, orientation='h',
        marker_color=top_colors,
        text=[f'{c:+.6f}' for c in top_coef],
        textposition='outside',
    ))
    fig.update_layout(
        title=dict(
            text=f'LEAR Lasso 系数 — Top {top_n} 特征'
                 f'<br><sup>红色=正相关, 蓝色=负相关, '
                 f'零系数={zero_count}/{total} ({zero_count/total*100:.0f}%) 被自动剔除</sup>',
            font=dict(size=16),
        ),
        xaxis=dict(title='Coefficient (Lasso)'),
        yaxis=dict(title='Feature', autorange='reversed'),
        height=500,
        margin=dict(l=200, r=60, t=80, b=40),
    )
    fig.add_vline(x=0, line_width=1, line_dash='dash', line_color='gray')
    fig.show()

    print('\nCoefficient statistics:')
    print(f'  Min: {coef.min():+.6f}')
    print(f'  Max: {coef.max():+.6f}')
    nonzero = coef[np.abs(coef) > 1e-10]
    if len(nonzero) > 0:
        print(f'  Non-zero range: [{nonzero.min():+.6f}, {nonzero.max():+.6f}]')
else:
    print('⚠️  LEAR 模型未加载或未训练。')


---
## Step 5: 跨模型特征重要性排名

用 `feature_importance_ranking()` 将 XGBoost 和 LEAR 放在一起比较。

**这张表回答的问题:**
- 哪些特征是两个模型都认为重要的？（共识）
- 哪些特征是一个模型重视、另一个忽略的？（分歧）
- 分歧的原因是模型结构差异还是数据特性？


In [ ]:
# ── 跨模型特征重要性排名 ─────────────
models_dict = {}
feature_names = None

if xgb is not None and hasattr(xgb, '_feature_cols'):
    models_dict['XGBoost'] = xgb
    feature_names = xgb._feature_cols

if lear is not None and hasattr(lear, '_feature_cols'):
    models_dict['LEAR'] = lear
    if feature_names is None:
        feature_names = lear._feature_cols

if len(models_dict) >= 2 and feature_names is not None:
    ranking = feature_importance_ranking(models_dict, feature_names)
    print('跨模型特征重要性排名 (Top 20):')
    print('=' * 70)
    display(ranking.head(20))
elif len(models_dict) == 0:
    print('⚠️  没有模型加载。')
else:
    print('⚠️  已加载模型不足 (<2)，可查看单模型排名:')
    name = list(models_dict.keys())[0]
    feat = feature_names or list(models_dict.values())[0]._feature_cols
    ranking = feature_importance_ranking(models_dict, feat)
    display(ranking.head(15))


---
## Step 6: 可视化对比 — 双模型特征重要性

把排名表变成双柱图，直观对比 XGBoost 和 LEAR 对每个特征的重要性判断。
使用归一化处理（各自除以最大值）以便在同一尺度上比较。


In [ ]:
# ── 双模型特征重要性对比图 ───────────
if 'ranking' in dir() and not ranking.empty:
    top_features = set()
    for model_name in ranking['model'].unique():
        top = ranking[ranking['model'] == model_name].head(10)['feature'].tolist()
        top_features.update(top)

    top_features = list(top_features)
    ranking_subset = ranking[ranking['feature'].isin(top_features)].copy()

    for model_name in ranking_subset['model'].unique():
        mask = ranking_subset['model'] == model_name
        max_val = ranking_subset.loc[mask, 'importance'].max()
        ranking_subset.loc[mask, 'importance_norm'] = (
            ranking_subset.loc[mask, 'importance'] / max_val if max_val > 0 else 0.0
        )

    pivot = ranking_subset.pivot_table(
        index='feature', columns='model', values='importance_norm', aggfunc='first'
    ).fillna(0)

    pivot['mean'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('mean', ascending=True).drop(columns='mean')

    fig = go.Figure()
    for col in pivot.columns:
        fig.add_trace(go.Bar(
            name=col, y=pivot.index, x=pivot[col], orientation='h',
            text=[f'{v:.2f}' for v in pivot[col]],
            textposition='outside',
        ))

    fig.update_layout(
        title=dict(
            text='跨模型特征重要性对比 (归一化)'
                 '<br><sup>XGBoost=feature_importances_, LEAR=|coef_|, 各自除以最大值</sup>',
            font=dict(size=16),
        ),
        xaxis=dict(title='Normalized Importance'),
        yaxis=dict(title='Feature', autorange='reversed'),
        barmode='group',
        height=400,
        margin=dict(l=180, r=40, t=80, b=40),
    )
    fig.show()
else:
    print('⚠️  排名数据不可用，请先运行 Step 5。')


---
## 讨论: 为什么不同模型对特征的重要性判断不同？

### 1. 模型结构差异

| 维度 | XGBoost (树模型) | LEAR (Lasso 线性模型) |
|------|-----------------|----------------------|
| 关系类型 | 自动捕捉非线性 + 交互 | 仅建模线性关系 |
| 特征选择 | 基于分裂增益 | L1 正则化自动剔除 |
| 交互效应 | 树的分支天然处理 | 需要手动添加交互特征 |
| 对量纲敏感度 | 低（树不必关心尺度） | 高（需要标准化） |

### 2. 典型的分歧模式

- **XGBoost 重视、LEAR 忽略**: 可能是有交互作用的特征
  - 例: `hour_of_day` 单个特征预测能力不强，但 `hour x lag_24h` 很有用
  - XGBoost 能把 hour 在多个分支中组合，赋予高重要性
  - Lasso 如果没加交互项，hour 的系数可能接近零

- **LEAR 重视、XGBoost 忽略**: 通常是强线性相关特征
  - 例: `lag_24h_price` 与当前电价高度线性相关
  - Lasso 会给它一个大系数（直接比例关系）
  - XGBoost 可能把它用在多个深度节点中，分散了重要性

### 3. 实际含义

两种模型各有所长。在电力交易中：
- **XGBoost**: 预测精度通常更高，适合做决策依据
- **LEAR**: 系数透明可解释，适合做风控和监管合规
- **最佳实践**: 同时使用两个模型，当它们预测一致时信心更强，
  出现分歧时深入分析原因


---
## 知识延伸

### SHAP 的数学理解

Shapley 值是合作博弈论中的概念，
衡量每个「玩家」（特征）在联盟（特征组合）中的边际贡献。
它满足三个理想性质:

1. **效率性 (Efficiency)**: 各特征贡献之和等于总预测减去平均
2. **对称性 (Symmetry)**: 两个特征对每个子集贡献相同，则 Shapley 值相等
3. **可加性 (Additivity)**: 多个游戏的 Shapley 值可以相加

### 局限性与注意事项

- SHAP 计算成本高: TreeExplainer 随特征数指数增长（优化后近似）
- SHAP 值可能产生误导：当特征高度相关时，贡献分配不稳定
- 线性模型的系数仅在标准化后才有可比性
- 不要混淆「相关性」与「因果性」: SHAP 值反映的是模型学到的模式，
  不一定是物理世界的因果机制


---
## 思考题

1. 观察 XGBoost 的 SHAP waterfall 图：哪个特征对当前样本的贡献最大？
   它的 Shapley 值是正是负？结合实际电力负荷的物理意义，
   这个特征为什么在这个时刻有这么大的影响？

2. LEAR Lasso 模型有多少个特征的系数被压缩到了零？
   这些被剔除的特征真的「没有用」吗？
   如果用 Ridge (L2) 代替 Lasso (L1)，零系数的比例会如何变化？

3. 比较 XGBoost 和 LEAR 的特征重要性排名：哪些特征是两者都排在前 5 的？
   哪些是一个排前 5 另一个排后 5 的？造成这种分歧的根本原因是什么？

4. 修改 `sample_idx`，选择另一个样本查看 waterfall 图：
   不同样本的 SHAP 值排序是否一致？
   什么情况下特征贡献的排序会发生显著变化？
   （提示：考虑不同时间段、不同负荷水平的预测场景）

5. （进阶）如果你要在电力交易场景中向监管机构解释一个高价预测：
   你会用 XGBoost 的 SHAP 图还是 LEAR 的系数表？为什么？
   哪个更能支持「报价合理性」的论证？
